# Writing a workflow for the framework: specs and bindings

This notebook is for the authors of workflow packages, such as esssans.
A package that offers a workflow to the framework writes two things, both against `ess.spec`:

- a **spec**: the workflow's name, version, and two pydantic models: its params and its outputs. Apps build forms from it, the framework checks requests against it, and records name it. It is all that anyone outside the package sees.
- a **binding**: the code that computes the spec. Apart from the validators of the models and of the types in them, it is the only code of the package that the backend runs.

`ess.spec` needs only pydantic; a module that defines specs whose data are scipp objects also imports scipp.
`ess.spec.pipeline` adds sciline.
A workflow package depends on `ess.spec`, not on `ess.apps`, the client and backend that run it.

The thread: a reduction of one run, its spec and binding, then the same reduction over several runs, summed as they arrive by a *driver*.
At each step, what the framework gives the binding, and what the binding's author promises in return.

## 0. The workflow you have: a sciline pipeline

The toy SANS reduction of `toy_sans.py` stands in for esssans.
A run's file holds its counts per pixel and its monitor count.
The detector is one row of four pixels, at x = 0, 1, 2, and 3 mm.
`load` stands for loading and correcting a run, the costly step.
A pixel's Q grows with its distance from the beam centre.
The reduction orders the pixels by that distance and sums them into Q bins (the numerator), divides by the monitor count times the sample's transmission (the denominator), and subtracts the can's curve from the sample's.

The pipeline's keys for the settings a user gives are classes that a spec can declare, and a run's file is a *file field*, defined in `toy_sans.py` as:

```python
class Reduction(BaseModel):
    bins: int = Field(default=2, gt=0)

class BeamCentre(BaseModel):               # in mm; (0, 0) is the nominal centre
    x: float = 0.0
    y: float = 0.0

class TransmissionFraction(BaseModel):     # a transmission, typed in or computed
    value: float = Field(gt=0, le=1)

RunFile = Annotated[Path, File()]          # a run's raw file, as its local path

class IofQ(BaseModel):                     # the result: one value per Q bin
    model_config = ConfigDict(arbitrary_types_allowed=True)

    intensity: sc.Variable
```

A parameter that takes a file is a file field, typed `Annotated[Path, File()]`: it holds the file's local path (section 1).
The transmission is a parameter of its own, not a field of a group with the sample run: the summed sample runs have one transmission (section 4), as in esssans.
The pipeline takes a run's file at a key of its own, `Filename[SampleRun]` or `Filename[CanRun]`, as esssans does, and a provider takes a plain path:

```python
def read(path: Filename[RunType]) -> RunData[RunType]:
    return RunData[RunType](json.loads(path.read_text()))
```

The reduction uses the beam centre as a scipp vector, `BeamCentrePosition`.
A provider makes it from the group:

```python
def beam_centre_position(centre: BeamCentre) -> BeamCentrePosition:
    return BeamCentrePosition(sc.vector([centre.x, centre.y, 0.0], unit='mm'))
```

So the pipeline's key is the group, which a spec can declare (section 1).
The result is a group too, `IofQ`, so that another workflow can take it, at a field typed `Referenced[IofQ]` (section 1).

`toy_sans.write_runs` writes the toy runs as small files:

In [ ]:
import tempfile
from pathlib import Path

import toy_sans
from toy_sans import (
    BeamCentre,
    CanRun,
    Filename,
    IofQ,
    Reduction,
    RunFile,
    SampleRun,
    TransmissionFraction,
    make_pipeline,
)

folder = tempfile.TemporaryDirectory()
paths = toy_sans.write_runs(Path(folder.name))
print(paths[101].read_text())

workflow = make_pipeline({})
workflow[Filename[SampleRun]] = paths[101]
workflow[Filename[CanRun]] = paths[110]
workflow[TransmissionFraction] = TransmissionFraction(value=0.8)
workflow[Reduction] = Reduction(bins=2)
workflow[BeamCentre] = BeamCentre()  # the nominal centre
workflow.compute(IofQ)

## 1. The spec of one run

The spec declares two models:

| Model | Holds | In a request | When the workflow runs |
|---|---|---|---|
| params | everything the workflow takes: settings, files, data | each value written inline, a file, or a reference to an output | the values, files with their local paths, and the data the references name |
| outputs | results | (none) | the data the workflow returns |

Each field of the params model is a *parameter*.
A request gives the value of each field, at any depth, in one of three forms:
*written inline*, the value itself;
a *file*, such as `file(run=101)`, which the backend resolves when the request is submitted;
or a *reference to an output*, which names an output of a record or of an accumulator.
The record of a request stores what the request gives in *JSON form*, what it dumps to as JSON: the *stored form*.
The type of a field says which forms it takes:

| Type | Example | A request writes |
|---|---|---|
| a scalar | `int`, `float`, `str`, `bool` | its JSON value |
| a file field | `RunFile`, `Annotated[Path, File(kind='mask')]` | a file |
| a group | `Reduction`, `BeamCentre`, `TransmissionFraction` | an object of its fields |
| a union of groups | `ByMonitor \| ByTime` (below) | an object of one alternative's fields, with its tag |
| `Referenced[T]` | `Referenced[IofQ]`, the parameter of `FORWARD` in `toy_sans.py` | a reference to an output that fits `T` |
| a union of members written inline and members `Referenced[...]` | `BeamCentre \| Referenced[BeamCentre]` | what one of its members takes |
| `X \| None` | `range: WavelengthRange \| None` (section 2) | what `X` takes, or `None` |
| `list[X]` | `list[RunFile]` (section 4) | a list of elements, each as for `X` |

These are the *field types*.
Each field of the params model, or of a group nested to any depth, is of a field type.
`WorkflowSpec` refuses any other type, such as `tuple[float, float]`, `dict[str, float]`, `int | str`, or a scipp variable.
`Referenced[T]` is `Annotated[T, REFERENCED]`, so pydantic, the binding, and any direct use of the params model see `T`.
Likewise a file field is `Annotated[Path, File(...)]`, so they see the file's local path.

A *group* is a pydantic model, such as `Reduction`, written as an object of its fields.
A field left out takes its default.
A *scalar* is an `int`, `float`, `str`, `bool`, or `datetime`, or an enum or a `Literal` whose values are of these types, possibly with constraints, such as `Annotated[float, Gt(0)]`.
Groups nest: a field of a group may be of any field type, such as a group, a union of groups, a list, or `Referenced[T]`.
A value of several parts is a group: `range: WavelengthRange`, not `limits: tuple[float, float]`.

A group with a field of another type, such as the scipp variable `intensity` of `IofQ`, has no form for a request to write.
So a field that takes an `IofQ` is typed `Referenced[IofQ]`, and `WorkflowSpec` refuses an `IofQ` at a field of any other type.
The framework does not look inside the `T` of `Referenced[T]`, so a field of `T` may be of any type, except a `Path` or `Referenced[...]`: no output holds a file.
`Referenced[T]` may type any field of the params, at any depth, but no field of the outputs.
A reference fits the field by the class of `T`, a group.
A field typed `float` or a scipp variable would fit every output of its type, so a picker of records would list unrelated outputs.
A group's class says what the value is, such as `BeamCentre` or `IofQ`.
So an array is a field of a group.
A value that may be absent is typed `X | None`, such as `mask: Referenced[PixelMask] | None`, with the array a field of `PixelMask`.
A list is `[]` when it has no elements, so it is not typed `list[X] | None`, and its element is not typed `X | None`.
By convention a parameter is a group, which a generated form shows as one tab and `PipelineBinding` sets as one key (section 2).
A parameter of another type, such as `bins: int` or a file field, needs `keys=` in the binding.
[ess.spec._workflow_spec](../../../packages/essspec/src/ess/spec/_workflow_spec.py) states the rules.

The settings, `reduction`, `beam_centre`, and `transmission`, are in a base model of their own, `IofQSettings`, which the spec of several runs shares (section 4):

In [ ]:
from pydantic import BaseModel, Field

from ess.spec import Referenced, WorkflowSpec


class IofQSettings(BaseModel):
    reduction: Reduction = Field(default_factory=Reduction)
    beam_centre: BeamCentre | Referenced[BeamCentre] = Field(default_factory=BeamCentre)
    transmission: TransmissionFraction | Referenced[TransmissionFraction]


class IofQParams(IofQSettings):
    sample: RunFile
    can: RunFile


class IofQOutputs(BaseModel):
    iofq: IofQ


IOFQ = WorkflowSpec(
    name='toy-sans-iofq',
    version=1,
    title='I(Q)',
    description='I(Q) of one sample run, minus that of one can run.',
    params=IofQParams,
    outputs=IofQOutputs,
)

Rules of the spec, each with its reason:

- **The spec says what a value is, by its type, never where it lives.** A request names a file or another record's output; the binding gets the data read, a file as its local path. A parameter that takes a file is a file field, typed `Annotated[Path, File(...)]`. A request writes a file as `file(...)`, which the backend resolves when the request is submitted, so that the file source checks access and each file a request reads has an identity in its record. The keywords of `File(...)` name the metadata of the files that fit, such as `MaskFile = Annotated[Path, File(kind='mask')]`; `File()` names `kind='raw'`, so `RunFile` takes raw files. The binding never sees the identity, only the path. A `Path` without `File(...)` is refused.
- **A choice between ways of obtaining a value is a union of groups.** It is pydantic's discriminated union, in which each alternative is a group with a *tag* field, a `Literal` of one string, its own. A union that holds a group holds groups only, since a value of another type has no tag. A union may type any field, at any depth, and a generated form shows it as a choice:

  ```python
  class ByMonitor(BaseModel):
      mode: Literal['monitor'] = 'monitor'

  class ByTime(BaseModel):
      mode: Literal['time'] = 'time'
      seconds: float = Field(gt=0)

  class Params(BaseModel):
      normalization: Annotated[ByMonitor | ByTime, Field(discriminator='mode')] = ByMonitor()

  # the params of a request
  {'normalization': {'mode': 'time', 'seconds': 2.0}}
  ```

  The request writes the union inline, with its tag, `mode`: the tag says which alternative's fields the object has. So the union is discriminated by the name of its tag field, not by a callable, and the tag has no alias. A reference to an output fits a union only through a member `Referenced[T]`, which has no tag. If an output could fit a tagged alternative, it would carry the tag of one alternative, so the workflow that computes it would have to know the specs that take it. So a value that another workflow computes is a group at a member `Referenced[...]` (next rule), and the discriminator sits on the union of the members written inline: `Annotated[ByMonitor | ByTime, Field(discriminator='mode')] | Referenced[ComputedNormalization]`. A field that only one mode uses belongs to that mode's group, not next to a selector between modes such as `mode: Literal[...]`, so that a request cannot hold a value that means nothing. A `bool` field switches one optional part of a group whose fields all have defaults, such as `limit_range: bool` next to `start` and `stop` (section 2).
- **The spec types each field that a reference fills `Referenced[T]`, and the request chooses within the union that the spec declares.** One user types in a beam centre, and another computes it with a first reduction. So `beam_centre` is typed `BeamCentre | Referenced[BeamCentre]`: a request writes a `BeamCentre` inline, or gives a reference to an output of type `BeamCentre`, such as the `centre` that `FIND_CENTRE` in `toy_sans.py` outputs. The binding gets a `BeamCentre` either way. A field without a member `Referenced[...]`, such as `reduction`, takes no reference, and a field typed `Referenced[T]` alone takes nothing written inline. A value that the workflow uses as a scipp object, such as the beam centre's vector, is a group in the spec and converted by a provider, so that a user can type it in. A value that one workflow computes and another takes is a group too: `FORWARD` in `toy_sans.py` takes the `IofQ` that `IOFQ` outputs, `curve: Referenced[IofQ]`. A field may also take a file or a reference to an output, such as a normalization that some users read from the catalogue and others compute, `normalization: PublishedNormalization | Referenced[ComputedNormalization] | None`, with `PublishedNormalization` a file field: the binding gets the file's local path or the group, and reads the file itself.
- **A class says what a value is, and is a parameter's key in a sciline pipeline.** No type in the params or the outputs is a `NewType`, at any depth: `NewType`s are sciline keys, which live in the workflow package and in the binding, never in the spec. A file field is a `Path`, which names no role, so the binding gives each file parameter the workflow's key with `keys=`. `PipelineBinding` refuses two parameters with one key, unless `keys=` names another key for one. Section 2 shows how a spec's values reach a pipeline with keys of its own.
- **Defaults live in the models only.** The binding gets every value filled in, so a default in the binding would never be used. A value *round-trips* when its JSON form reads back as the value. A default is a value, never a reference to an output or a file: what a request may write at the field, which round-trips and holds no `@ref` object at any depth. A file field, and a field that only a reference fills, has no default but `None`, or `[]` for a list. It is the value that validating it gives, since pydantic does not validate a default: a group's default is an instance, `Reduction()`, not the object of its fields, and an enum's a member, not its value.
- **A request that leaves out a parameter, or a field of a group, so that it takes its default, never skips a correction that a correct reduction needs.** The default of `reduction` is a setting. The can run has no default, so a request that leaves it out is refused instead of computing I(Q) without the can subtracted. A correction that a correct reduction needs, but that a request may still go without, is typed `X | None` with no default, such as `direct_beam: DirectBeamFile | None` in esssans (section 2), so the request writes `None` and its record says so: the output is I(Q) either way, and the record's params say which corrections were applied.
- **A check of one value belongs to its type.** `Reduction` refuses `bins=0`. Each parameter is checked alone, by its own field: a template, a stage call, or a push gives some values only ([README, Templates](../README.md#templates)). So `WorkflowSpec` checks this rule: The params model has no field validators, serializers, computed fields, aliases, or `Field(exclude=...)`, and no `default_factory` that takes the other values. A table carries only length constraints, such as `Field(min_length=2)`, since an accumulator takes it a few rows at a time (section 4). The params model may have model validators, but the framework never calls them. A group may have field and model validators of any mode, but no serializers, computed fields, aliases, or `Field(exclude=...)`, since the record stores a group as the object of its fields. The framework validates a group when the request is submitted, so that an error shows before anything runs: every validator runs then, such as a model validator that compares two fields, with the local path of each file that the group holds, and the record stores the group as validated. A group that holds a reference to an output, at any depth, is validated when the request runs instead: the output may not exist at submission, and the group's validators see the data read. A failure then fails the record, or stops the accumulator for a push. In Python, a request gives such a group as a dict, since pydantic refuses a reference in an instance of the group. A file field carries no validators: what it would check of a file is its metadata, which the file source checks when the request is submitted, or its content, which is the workflow's to read.
- **A check across parameters is the binding's.** A binding may run the model validators, `IofQParams.model_validate(dict(params))`; the check then fails when the request runs.
- **Name and version identify the spec.** Records name them. `version` is the version of the interface: it changes with any change a consumer could observe, in the params or outputs models, or in what the workflow computes. The optional field `code_revision` names the code itself; records do not carry it.

An app reads the spec, not the package's code.
`serialize()` gives the schema of the params.
A file names in `metadata` the metadata of the files that fit, and the reference at each field typed `Referenced[T]` names in `dataType` the type of data that fits, from which a picker of files or records is built; a group that a request writes has its JSON Schema, from which a form is built.
`beam_centre` takes either, so its schema is a `oneOf` of the two:

In [ ]:
serialized = IOFQ.serialize()
properties = serialized.params_schema['properties']
properties['beam_centre'], properties['sample']

A request lives in two phases.
In the *request phase*, before any data is read, the framework validates each group but those that hold a reference to an output, resolves each file, and checks that the data each reference to an output names fit the type of its field.
In the *call phase*, the params model validates the stored form, with each reference replaced by its data and each file given its local path, so every field holds the type it declares.
It fails only for the data of a reference to an output, or for a group that holds one.
Used directly, the params model types the call phase: it refuses what cannot run, and takes no reference to an output:

In [ ]:
from pydantic import ValidationError

from ess.spec import OutputRef

try:
    IofQParams(
        reduction={'bins': 0},
        sample=paths[101],
        can=paths[110],
        transmission={'value': 0.8},
    )
except ValidationError as error:
    print(error)
try:
    toy_sans.ForwardParams(curve=OutputRef(record='r1', output='iofq'))
except ValidationError as error:
    print(error)

## 2. The binding of one run

A binding is what the backend calls to compute a spec's requests.
The simplest binding is a function.
It takes an instance of the params model and returns one of the outputs model.
It gets every default filled in and every reference read:

In [ ]:
from ess.spec import function_binding


def iofq(params: IofQParams) -> IofQOutputs:
    workflow = make_pipeline({})
    workflow[Reduction] = params.reduction
    workflow[BeamCentre] = params.beam_centre
    workflow[TransmissionFraction] = params.transmission
    workflow[Filename[SampleRun]] = params.sample
    workflow[Filename[CanRun]] = params.can
    return IofQOutputs(iofq=workflow.compute(IofQ))


draft = function_binding(IOFQ, iofq)
params = IofQParams(
    sample=paths[101],
    can=paths[110],
    transmission=TransmissionFraction(value=0.8),
)
draft.compute(params)

A request with every value given, a *plain request*, goes through `compute(params)`.
The spec is generic over its models, so a type checker checks `iofq` against `IOFQ`.

For a sciline pipeline, `PipelineBinding` does the same.
Each group sets the key of its class, here `Reduction`, `BeamCentre`, and `TransmissionFraction`, and each output is computed as the key of its type, here `IofQ`.
A file field is a `Path`, which names no role, so `keys=` gives each file parameter the workflow's key of its path; `PipelineBinding` refuses a file parameter without one.
`make_pipeline` gets the fixed values by name, so that the graph may depend on them, and may return a pipeline that it holds, since the binding copies it:

In [ ]:
from ess.spec.pipeline import PipelineBinding

single = PipelineBinding(
    IOFQ,
    make_pipeline,
    keys={'sample': (Filename[SampleRun],), 'can': (Filename[CanRun],)},
)
single.compute(params)

In `toy_sans.py` the spec's classes are the pipeline's keys, since the pipeline was written for the spec.
An existing workflow has keys of its own, such as `Filename[SampleRun]` and `BeamCenter` in esssans.
A spec module cannot import workflow code, so it cannot declare these types.
Nor would it: keys such as `Filename[SampleRun]`, and `NewType`s in general, are sciline keys, which live in the workflow package and in the binding, never in the spec.
The spec names its parameters for their role, `sample` and `can`, and types each a file field; it never names `SampleRun` or `BackgroundRun`, which are esssans' run types.
A binding takes the spec's values to the pipeline's keys in one of three ways:

```python
# 1. keys=: the pipeline's key for one value, which it takes or computes as the
#    spec types it, here the files, as paths, and the output `curve: sc.DataArray`
#    of a spec of its own
class CurveOutputs(BaseModel, arbitrary_types_allowed=True):
    curve: sc.DataArray

IOFQ_CURVE = WorkflowSpec(name='sans-iofq-curve', version=1, title='I(Q) curve',
                          description='...', params=IofQParams, outputs=CurveOutputs)
PipelineBinding(
    IOFQ_CURVE,
    make_pipeline,
    keys={
        'sample': (Filename[SampleRun],),
        'can': (Filename[BackgroundRun],),
        'curve': (BackgroundSubtractedIofQ,),
    },
)

# 2. an adapter provider: takes a group of the spec, returns a key of the pipeline,
#    or the other way round for an output
def beam_center(centre: BeamCentre) -> BeamCenter:
    return BeamCenter(sc.vector([centre.x, centre.y, 0.0], unit='mm'))

def iofq(curve: BackgroundSubtractedIofQ) -> IofQ:
    return IofQ(intensity=curve)

def make_pipeline(fixed: Mapping[str, Any]) -> sciline.Pipeline:
    pipeline = sans_workflow()
    pipeline.insert(beam_center)
    pipeline.insert(iofq)
    return pipeline

# 3. a provider of the workflow that takes the spec's group, as in toy_sans.py
def beam_centre_position(centre: BeamCentre) -> BeamCentrePosition:
    return BeamCentrePosition(sc.vector([centre.x, centre.y, 0.0], unit='mm'))
```

`keys=` fits a value that the pipeline takes or computes as the spec types it, such as a file's path, an output typed `sc.DataArray`, or a group that the pipeline takes at a key of its own.
An adapter provider fits a value that the pipeline takes or computes in another shape, such as a field of a group, a scipp object made from a group, or a group made from a scipp object.
Defaults belong to the spec: `make_pipeline` holds no value at a key that a parameter's key leads to, since the held value replaces the provider that would read the parameter.
`make_pipeline` replaces such a value with a provider, as `pipeline.insert(beam_center)` above replaces a `BeamCenter` that `sans_workflow()` may hold, so that the request's `beam_centre` takes effect.
`PipelineBinding` cannot see such a value in general, but it refuses one held at a key of a union, which one member sets and the others do not.
Most workflows start with 1 and 2.
The goal is 3: the workflow's providers take the spec's models, and the adapters are gone.
A group is one key, so a provider takes it apart, as `beam_centre_position` reads `centre.x` and `centre.y`.
A union of groups has a key for each member, `(ByMonitor, ByTime)`, and `PipelineBinding` sets the key of the member nearest to the value's class, the first class of `type(value).__mro__` that is a member.
A union whose members are of one class, such as `BeamCentre | Referenced[BeamCentre]`, is that class to the binding, with one key.
Where a provider leads from one alternative's key to the other's, such as `find_centre(settings: FindCentre) -> GivenCentre` for the beam centre of [ADR 0008](../../../packages/essspec/docs/developer/adr/0008-a-choice-between-ways-is-a-union-of-groups.md), typed `GivenCentre | FindCentre`, one graph serves both alternatives.
Where the alternatives need different providers for one key, `make_pipeline` branches on the fixed value: a union is never a blank, since its key depends on its value.

`X | None` means *without X*: apply no mask, no direct-beam correction.
An app shows it as a toggle, so a toggle that is off must read as what `None` does.
Any field may be typed `X | None`, at any depth.
A choice between modes is a union of groups.
An optional part with fields that need values is one group typed `X | None`: `range: WavelengthRange | None`, not `start: float | None` and `stop: float | None`, so that a request gives both bounds or neither.
A `bool` field switches one optional part of a group whose fields all have defaults, such as `limit_range: bool` next to `start` and `stop`, both with defaults.
A value obtained in one of several ways that a request writes inline is a union that names each way, not `None` for one of them.
An app shows the union as a choice:

```python
mask: Referenced[PixelMask] | None = None  # [ ] Mask
direct_beam: DirectBeamFile | None         # [x] Direct beam  [run 60340 ▾]
beam_centre: BeamCentre = BeamCentre()     # the nominal centre, not BeamCentre | None
normalization: Annotated[                  # Normalization: (•) by monitor  ( ) by time
    ByMonitor | ByTime, Field(discriminator='mode')
] = ByMonitor()                            # not ByTime | None, whose toggle, off,
                                           # would read as "no normalization"
```

`DirectBeamFile` is a file field, `Annotated[Path, File()]`, the direct-beam run's file.

Each value of a union sets one key, but `None` has no key to set.
So a parameter typed `X | None` gets `keys=` `()`, and `make_pipeline` says what `None` means, in a pipeline it makes for this call:

```python
def make_pipeline(fixed: Mapping[str, Any]) -> sciline.Pipeline:
    pipeline = sciline.Pipeline(providers)  # never a pipeline it keeps: the value would stay
    if fixed['direct_beam'] is None:
        pipeline[sans.DirectBeam] = None    # esssans' key; its providers take None: no correction
    else:
        pipeline[sans.DirectBeamFilename] = fixed['direct_beam']
    return pipeline

PipelineBinding(IOFQ, make_pipeline, keys={'direct_beam': ()})
```

Each binding takes its spec, and the backend finds a request's binding by the spec's name and version.

**Promise: a binding modifies none of the values it gets, and returns no output that shares memory with one.**
The framework passes the values it holds without a copy, such as another record's output that a request references.
A binding that modified one, or returned a view of one, would change another record's value.
`forward` in `toy_sans.py` returns `params.curve.intensity['Q', 0].copy()` for this reason.

## 3. Stages: compute the fixed part once

When a user tunes `bins`, the runs stay the same.
The client then keeps a *stage*: a template whose fixed values stay, with *blanks*, the parameters left open (see [client-authors.ipynb](client-authors.ipynb)).
The backend calls `make_stage(fixed, blanks)` once, and the returned callable once per call through the stage, with the values of the blanks.
Both take values by name, each as its type, whether the request wrote it inline or gave it by reference: a group as the model, with the data of each reference read, and a file as its local path.
A parameter in neither, here `beam_centre`, takes its default.
`PipelineBinding` cuts the pipeline at the keys of the blanks with `sciline.Stage`, and computes what does not depend on them once.
So a blank needs one key: neither a union of several classes, whose key may depend on its value, nor a parameter typed `X | None`, which has none, is a blank of a `PipelineBinding`.
Files that vary independently are separate parameters, not fields of one group.
`sciline.Stage` decides what it computes again on each call by the graph alone: all that depends on the key of a blank.
A blank is its own key, and a group is one key, so a blank group of the sample and the can would load the can again on every call.
A group of files fits files that always change together, such as the two channels of a polarized run; a provider of `make_pipeline` may take the group apart into the workflow's keys.
`beam_centre`, typed `BeamCentre | Referenced[BeamCentre]`, has one key, so it may be a blank.
This stage tunes `reduction`:

In [ ]:
fixed = {
    'sample': paths[101],
    'can': paths[110],
    'transmission': TransmissionFraction(value=0.8),
}
call = single.make_stage(fixed, ('reduction',))
toy_sans.LOADED.clear()
for bins in (1, 2, 4):
    print(bins, call({'reduction': Reduction(bins=bins)}).iofq.intensity.values)
print('runs loaded:', len(toy_sans.LOADED))

Two loads for three calls: the sample run and the can run, once each.
A function binding has no `make_stage`, since a stage over a function would hold nothing; the backend refuses a stage over it when the stage is made.

**Promise: a call through a stage gives the outputs of the plain request** (the *caching* symmetry).
The framework records a call through a stage as its plain request, so the record must not depend on how it was computed.
The `sciline.Stage` keeps this promise, given a pure pipeline.
`ess.spec.testing` checks it by computing both, and also checks that no output shares memory with a value the binding got:

In [ ]:
from ess.spec.testing import check_caching

check_caching(single, fixed, [{'reduction': Reduction(bins=n)} for n in (2, 1, 2)])

## 4. The spec of several runs: tables

Many reductions combine runs: here, several sample runs and several can runs, summed.
A *table* is a list parameter whose *rows* an accumulator takes a push at a time, such as the sample runs.
A row here is a run's file, as its local path; a row that holds more than one value is a group.
The multi-run spec has a params model of its own.
Each table is a list of the single-run type, here `list[RunFile]` for both.
The other parameters, `reduction`, `beam_centre`, and `transmission`, are those of `IofQSettings`, shared with `IofQParams`.
`tables` names the tables; a list parameter is a table only where `tables` names it:

In [ ]:
class IofQMultiParams(IofQSettings):
    sample: list[RunFile] = Field(min_length=1)
    can: list[RunFile] = Field(min_length=1)


IOFQ_MULTI = WorkflowSpec(
    name='toy-sans-iofq-multi',
    version=1,
    title='I(Q) of summed runs',
    description='I(Q) of the summed sample runs, minus that of the summed can runs.',
    params=IofQMultiParams,
    outputs=IofQOutputs,
    tables=('sample', 'can'),
)
IOFQ_MULTI.tables

Rules of tables:

- **Each table is a list of the single-run type**, with at least one row, `Field(min_length=1)`. A table has no default, so a request gives every table, `[]` for one that may have no rows, and a template that forgets a table is refused; `WorkflowSpec` refuses a table with a default. The validators and constraints of the element type check each row; the table carries only length constraints, which the backend checks on all rows. The other parameters and the outputs model are those of `IOFQ`.
- **With one row in each table, the two specs agree**: a request of `IOFQ_MULTI` is then a request of `IOFQ`. `check_one_row` checks this (section 6).
- **Each row is written inline, a file, or a reference to an output, as the spec types it.** A `RunFile` row is a file; a row of a group is written inline as an object of its fields; a row of `list[Referenced[T]]` is a reference to an output; and a row of `list[X | Referenced[X]]` is either.
- **A row of a table of runs is a file.** A value that differs per run, such as a motor position, is in the run's file. A value that differs between runs, such as a thickness, makes the runs separate requests; a value shared by all runs of a request is a parameter, such as the transmission of the sample runs. Tables are independent of each other; which runs go together is the app's choice.
- **The rows of a table are distinct**: a row is in a table once. Rows are compared by their stored form: a run by its file's identity, however the request named it, and a reference to an output by its record and output. Two references to equal data are two rows.
- **A setting is tuned without loading the runs again.** A blank is a parameter, and a parameter is one key of the pipeline. The transmission is a parameter of its own, so a stage of `IOFQ` that tunes it has `transmission` as its blank, and loads no run again.
- **Structure below a row is the binding's**, such as detector banks. The framework sees rows only.
- **A multi-run spec may have outputs of its own, or no single-run spec**, such as a stitch of all angles, or a sum of counts over the runs of a scan. It is written the same way: `angle: list[RunFile] = Field(min_length=1)`, with `WorkflowSpec(..., tables=('angle',))`.

## 5. A driver: summing runs as they arrive

When runs arrive one at a time, a user wants the sum so far after each, without reducing the earlier runs again.
The client's *accumulator* takes one *push* at a time, each one or more rows of one or more tables (see [client-authors.ipynb](client-authors.ipynb)).
Behind it is a *driver*: an object with `push(table, rows)` and `outputs()`, which the workflow package writes for one spec.
A push names its table and gives its rows, each as its type: here a run's local path.
Only a driver accumulates: the framework never decides how rows combine.

This driver sums each run's numerator and denominator per table, and computes I(Q) once from the sums.
It cuts the one-run pipeline with `sciline.Stage` twice: at the key of a table, `Filename[SampleRun]` or `Filename[CanRun]`, to compute one run's numerator and denominator, and at the four sums, to compute I(Q).
It gets the fixed values that are not tables, here the groups `reduction`, `beam_centre`, and `transmission`, when it is made.
It takes the graph and the keys from the single-run binding, `single.pipeline(fixed)` and `single.keys`, so that it sets no key by hand:

In [ ]:
from collections.abc import Mapping, Sequence
from typing import Any

import sciline
import scipp as sc

from toy_sans import Denominator, Numerator


class IofQDriver:
    def __init__(self, fixed: Mapping[str, Any]) -> None:
        workflow = single.pipeline(fixed)
        self._contribute = {
            table: sciline.Stage(
                workflow,
                inputs=single.keys[table],
                outputs=(Numerator[run], Denominator[run]),
            )
            for table, run in (('sample', SampleRun), ('can', CanRun))
        }
        sums = [
            key[run] for key in (Numerator, Denominator) for run in (SampleRun, CanRun)
        ]
        self._finalize = sciline.Stage(workflow, inputs=sums, outputs=(IofQ,))
        self._sums: dict[str, dict[Any, sc.Variable]] = {}  # per table, by key

    def push(self, table: str, rows: Sequence[Any]) -> None:
        stage = self._contribute[table]
        (key,) = single.keys[table]
        for row in rows:
            parts = stage.apply({key: row})
            old = self._sums.get(table)
            self._sums[table] = (
                parts if old is None else {k: old[k] + v for k, v in parts.items()}
            )

    def outputs(self) -> IofQOutputs:
        if empty := sorted(self._contribute.keys() - self._sums.keys()):
            raise ValueError(f'no rows in {empty} yet')
        sums = {k: v for parts in self._sums.values() for k, v in parts.items()}
        return IofQOutputs(iofq=self._finalize.apply(sums)[IofQ])


groups = {
    'reduction': Reduction(bins=2),
    'beam_centre': BeamCentre(),
    'transmission': TransmissionFraction(value=0.8),
}
driver = IofQDriver(groups)
toy_sans.LOADED.clear()
driver.push('sample', [paths[101]])
driver.push('can', [paths[110]])
print(driver.outputs().iofq.intensity.values)
driver.push('sample', [paths[102], paths[103]])
print(driver.outputs().iofq.intensity.values)
print('runs loaded:', len(toy_sans.LOADED))

Four loads for four runs: a push loads only its own rows.
Summing numerators and denominators apart, and dividing once, is what makes a sum of runs correct; a sum of normalized curves would not be.
The driver needs no spec, backend, or client, so a package uses and tests it as plain code.

The author of a driver promises, and the backend does not check:

- **`outputs()` gives the outputs of every row pushed so far**, however the rows were grouped into pushes, and whatever the order of pushes into different tables (the *arrival* and *order* symmetries). So each sum may depend on the fixed values and on one table only.
- **Within a table, the rows keep the order in which they were pushed.**
- **`push` does not modify a value it is given.** A value may be another record's output. A driver that adds in place copies the first contribution, so that adding never changes a value that another record returned. `IofQDriver` adds with `a + b`, which modifies nothing.
- **A driver may refuse a state**, such as one with no can row yet, by raising in `outputs()`. Later pushes may make the state valid.
- **A check across parameters or rows is the driver's.** A driver gets each value checked by its own field only; the framework runs no model validator of the params model.

In return, `outputs()` may share memory with what the driver holds: the backend is done with the outputs of a state before it pushes the next rows.

`check_arrival_and_order` takes the function that makes a driver, the fixed values that are not tables, and the rows of each table by its name.
It pushes the rows in several schedules, one row at a time and in groups, table by table and alternating, and reads between pushes.
It compares each state with what a new driver gives after one push per table, skipping states that the driver refuses.
It also fails if a push modifies the values it was given:

In [ ]:
from ess.spec.testing import check_arrival_and_order

tables = {
    'sample': [paths[run] for run in (101, 102, 103)],
    'can': [paths[110], paths[111]],
}
check_arrival_and_order(IofQDriver, groups, tables)

Both sides of the comparison go through the driver, so the check does not prove the outputs right.

A sum that depends on two tables would break the order symmetry: the can's sum would change with the sample rows pushed before it.
esssans has this case, since its pixel masks come from the sample run.
Its driver must give outputs that do not depend on the order of pushes, for example by keeping the can rows that a mask applies to; `check_arrival_and_order` catches a driver whose outputs do.

A workflow that needs every row at once, such as a fit over all runs, writes a driver that keeps its rows and computes over all of them in `outputs()`.
The outputs are the same as for a driver that sums; only the cost differs, and which to pay is the workflow's choice.

## 6. The binding of several runs

`DriverBinding` binds the multi-run spec to the function that makes a driver.
A plain request makes a driver, pushes the rows of each table in one push, and reads the outputs:

In [ ]:
from ess.spec import DriverBinding

multi = DriverBinding(IOFQ_MULTI, IofQDriver)
multi.compute(IOFQ_MULTI.params(**groups, **tables))

For an accumulator, the backend calls `make_accumulator(fixed, blanks)`, with the tables that the pushes fill as the blanks.
The binding makes a driver from the fixed values that are not tables, and pushes the rows of each table in `fixed`, such as a can run fixed by the client's template.
The backend then pushes each client push into the driver:

In [ ]:
fixed = {
    'reduction': Reduction(bins=2),
    'transmission': TransmissionFraction(value=0.8),
    'can': [paths[110]],
}
driver = multi.make_accumulator(fixed, ('sample',))
driver.push('sample', [paths[101]])
driver.outputs()

**Promise: with one row in each table, the binding of the multi-run spec gives the outputs of the single-run spec** (the *one-row* symmetry).
The specs agree: with one row in each table, a request of `IOFQ_MULTI` is a request of `IOFQ`, since `IofQMultiParams` shares the settings of `IofQParams` and each of its tables is a list of the single-run type.
`one_row` makes that request from a request of `IOFQ`:

In [ ]:
from ess.spec.testing import check_one_row, one_row

params = IofQParams(
    beam_centre=BeamCentre(x=1.25),
    sample=paths[103],
    can=paths[110],
    transmission=TransmissionFraction(value=0.6),
)
one_row(IOFQ_MULTI, params)

The package keeps the promise: the driver must compute what the pipeline computes.
`check_one_row` computes both and compares them; it is the only check against an independent reference:

In [ ]:
check_one_row(single, multi, params)

Each binding serves some of the client's actions, and the backend refuses the others when a stage or an accumulator is made:

| Binding | Plain request | Stage | Accumulator |
|---|---|---|---|
| `PipelineBinding` | `compute` | `make_stage` | refused |
| `DriverBinding` | `compute` | refused | `make_accumulator` |
| `function_binding` | `compute` | refused | refused |

## 7. What the framework gives the binding

The binding may rely on this:

| Call | When, and with what |
|---|---|
| `compute(params)` | once per plain request; an instance of the params model, with data read and defaults filled in, each value checked by its own field, and no model validator of the params model run; possibly from several threads at once |
| `make_stage(fixed, blanks)` | once per stage of a client, at its first call; every value but the blanks, by name; possibly from several threads at once |
| the callable it returns | once per call through the stage, with the values of the blanks, by name; possibly from several threads at once |
| `make_accumulator(fixed, blanks)` | once per accumulator; every value but the tables that the pushes fill, by name. `DriverBinding` calls its function that makes a driver here, and in `compute`, possibly from several threads at once |
| `driver.push(table, rows)` | once per client push, with its rows, data read; in the order logged; one thread at a time; never while the readers of the state before it are busy |
| `driver.outputs()` | once per state that someone reads; one thread at a time |

Each value the binding gets holds the type its field declares, whether the request wrote it inline, as a file, or as a reference to an output.
When a request is submitted, the backend resolves each file, validates each group that holds no reference to an output, with the local path of each file it holds, and checks the fit of each reference to an output by type. When the request runs, the params model validates the stored form with the data the references name, which also validates each group that holds a reference.
[framework-promises.ipynb](framework-promises.ipynb) traces these calls for one accumulator.

## Summary: what the author of a workflow promises

| Promise | Checked by |
|---|---|
| each field of the params, at any depth, is of a field type: a scalar, a file field, a group, a tagged union of groups, `Referenced[T]`, a union of members written inline and members `Referenced[...]`, `X \| None`, or `list[X]`; a group with a field of another type, such as an array, is at a field typed `Referenced[...]`; `Referenced[...]` types no output, and its `T` is a group that holds no `Path`; no type in the params or the outputs is a `NewType`; a file is a file field, `Annotated[Path, File(...)]`, and no `Path` is anywhere else | `WorkflowSpec` |
| a value that one workflow outputs and another takes is a group, at a field typed `Referenced[T]`, and an array is a field of a group; a value that the workflow uses as a scipp object is a group, converted by a provider, so that users can type it in | `WorkflowSpec` refuses a `T` that is not a group; nobody checks the rest |
| each parameter has a key of its own in the pipeline: a class of its own, or `keys=`, which a parameter that is not a group needs, such as a file field | `PipelineBinding` |
| files that vary independently are separate parameters, so that a stage computes again only what depends on its blanks | nobody |
| defaults live in the models only, and are values | `WorkflowSpec` checks that a default is a value; `PipelineBinding` refuses a value that `make_pipeline` holds at a key of a union; nobody checks the rest |
| `X \| None` means *without X*; a value obtained in one of several ways that a request writes inline is a union that names each way | nobody |
| a check of one value belongs to its type; a check across parameters is the binding's | `WorkflowSpec` checks the rule on the params model in section 1; nobody checks the rest |
| a new version when the params or outputs models change | nobody |
| the binding returns no output that shares memory with a value it gets | `check_one_row`, `check_caching` (numpy-backed values) |
| the binding modifies no value it gets | nobody, except for the values a push gets (`check_arrival_and_order`) |
| the pipeline is pure, so a call through a stage gives the outputs of the plain request | `check_caching` |
| the multi-run spec with one row per table takes the requests of the single-run spec, and gives its outputs | `check_one_row` |
| after each push, the driver's outputs are those of all rows pushed so far, in any grouping and in any order across tables | `check_arrival_and_order` |
| `push` modifies no value | `check_arrival_and_order` |
| structure below a row belongs to the binding | nobody |

A package registers its bindings through the entry-point group `ess.apps.workflows`.
The entry-point group is decided and not implemented; a notebook or app passes bindings to `local(..., bind=toy_sans.BINDINGS)`.

In [ ]:
folder.cleanup()